In [ ]:
import sys
import os

# Get the absolute path of the directory one level up (your project root)
project_root = os.path.abspath('..')

# Add it to the system path if it's not already there
if project_root not in sys.path:
    sys.path.append(project_root)

In [ ]:
import os
import time
import requests
from bs4 import BeautifulSoup
from langchain_chroma import Chroma
from langchain_ollama import OllamaEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from config.settings import Settings

In [ ]:
SITE_URL = Settings().sitemap_url
PERSISTENCE_DIR = Settings().mock_directory
HEADERS = {'User-Agent': 'RouhanMedicalProject/1.0'}

# Metadata Blacklist to ensure only medical knowledge is stored
BLACKLIST = [
    "references", "review date", "last reviewed", "reviewer", 
    "related topics", "find an expert", "patient handouts", 
    "clinical trials", "languages", "topic image"
]

AttributeError: 'Settings' object has no attribute 'mock_directory'

In [ ]:
print(f"📥 Fetching sitemap from {SITE_URL}...")
try:
    response = requests.get(SITE_URL, headers=HEADERS)
    response.raise_for_status()
except Exception as e:
    print(f"❌ Failed to fetch sitemap: {e}")

📥 Fetching sitemap from https://medlineplus.gov/sitemap.xml...


In [ ]:
soup = BeautifulSoup(response.content, "xml")
all_urls = [loc.text for loc in soup.find_all("loc")]

all_urls = all_urls[24:33]  # Limit to a smaller set for testing
print(f"🗺️ Found {len(all_urls)} URLs to process.")

# Initialize Vector Store
embeddings = OllamaEmbeddings(model=Settings().embedding_model)
vectorDB = Chroma(
    embedding_function=embeddings,
    persist_directory=,
)

# Improved splitter for medical context preservation
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, 
    chunk_overlap=150, # Increased overlap to catch warnings
    separators=["\n\n", "\n", ". ", " "]
)

In [ ]:
for url in all_urls:
    print(f"🔍 Processing URL: {url}")
    try:
        page_response = requests.get(url, headers=HEADERS)
        page_response.raise_for_status()
    except Exception as e:
        print(f"❌ Failed to fetch page: {e}")
        continue

    page_soup = BeautifulSoup(page_response.content, "html.parser")
    print(f"Title: {page_soup.title.string if page_soup.title else 'No Title'}")

    # Extract main content while preserving medical context
    content_div = page_soup.find("div", class_="main-content")
    if not content_div:
        print("⚠️ Main content not found, skipping.")
        continue
    print(f"Content length: {len(content_div.get_text())} characters")
    
    # Remove non-medical sections based on common patterns
    for section in content_div.find_all(["aside", "footer", "nav"]):
        section.decompose()
    
    text = content_div.get_text(separator="\n").strip()
    
    if not text:
        print("⚠️ No text extracted, skipping.")
        continue
    print(f"Extracted text length: {len(text)} characters")
    
    # Split text into chunks
    chunks = text_splitter.split_text(text)
    
    documents = []
    for chunk in chunks:
        metadata = {
            "source": url,
            "title": page_soup.title.string if page_soup.title else "No Title"
        }
        print(f"Processing chunk: {chunk[:100]}...")

        # Filter out blacklisted metadata
        if any(black in chunk.lower() for black in BLACKLIST):
            continue
        
        documents.append(Document(page_content=chunk, metadata=metadata))
    
    if documents:
        if os.path.exists(PERSISTENCE_DIR) and len(os.listdir(PERSISTENCE_DIR)) > 0:
            vectorDB.add_documents(documents)
            print(f"✅ Added {len(documents)} chunks to vector store.")
        else:
            print("⚠️ Persistence directory is empty or does not exist.")
    else:
        print("⚠️ No valid chunks to add after filtering.")

IndentationError: expected an indented block after 'else' statement on line 52 (1618141847.py, line 53)